In [1]:
import polars as pl
from Python import statcast, batter_features

years = [2025]

In [2]:
print(batter_features.BUILD_COLUMNS)
print(statcast.DISCIPLINE_COUNTS)
print(statcast.STRIKEOUT_EVENTS)
print(statcast.NON_PA_EVENTS)

('game_pk', 'game_date', 'batter', 'stand', 'p_throws', 'home_team', 'away_team', 'inning_topbot', 'events', 'description', 'type', 'zone', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom')
('InZone', 'OutZone', 'Swings', 'Chases', 'ZSwings', 'Contacts', 'ZContacts', 'OContacts')
frozenset({'strikeout', 'strikeout_double_play'})
frozenset({'pickoff_caught_stealing_2b', 'stolen_base_3b', 'pickoff_1b', 'stolen_base_2b', 'pickoff_caught_stealing_home', 'other_out', 'runner_double_play', 'caught_stealing_3b', 'stolen_base_home', 'pickoff_3b', 'passed_ball', 'pickoff_caught_stealing_3b', 'pickoff_2b', 'caught_stealing_home', 'wild_pitch', 'caught_stealing_2b', 'batter_interference'})


In [3]:
cross_check_extra = ["at_bat_number", "pitch_number", "inning"]
load_columns = list(dict.fromkeys(list(batter_features.BUILD_COLUMNS) + cross_check_extra))

raw = statcast.load_statcast_years(years, columns=load_columns)
raw.shape, raw.columns

((706571, 18),
 ['game_pk',
  'game_date',
  'batter',
  'stand',
  'p_throws',
  'home_team',
  'away_team',
  'inning_topbot',
  'events',
  'description',
  'type',
  'zone',
  'estimated_woba_using_speedangle',
  'woba_value',
  'woba_denom',
  'at_bat_number',
  'pitch_number',
  'inning'])

In [4]:
flagged = statcast.add_plate_discipline_flags(statcast.add_event_flags(raw))
flagged.select([
    "events", "description", "is_pa", "is_k", "is_bb", "is_hbp", "is_hr",
    "is_hit", "is_whiff", "is_called_strike",
    "is_in_zone", "is_out_zone", "is_swing", "is_contact", "is_chase",
]).head(10)

# Sanity: a strikeout must be a PA; a whiff must never also be scored as contact
flagged.select(
    (pl.col("is_k") & ~pl.col("is_pa")).sum().alias("k_without_pa"),
    (pl.col("is_whiff") & pl.col("is_contact")).sum().alias("whiff_and_contact"),
)

k_without_pa,whiff_and_contact
u32,u32
0,0


In [5]:
bat_team_check = flagged.with_columns(
    pl.when(pl.col("inning_topbot") == "Top")
      .then(pl.col("away_team"))
      .otherwise(pl.col("home_team"))
      .alias("bat_team")
)

# bat_team should always equal home_team or away_team, never null
bat_team_check.select(
    ((pl.col("bat_team") != pl.col("home_team")) & (pl.col("bat_team") != pl.col("away_team"))).sum().alias("bad_bat_team"),
    pl.col("bat_team").is_null().sum().alias("null_bat_team"),
)

bad_bat_team,null_bat_team
u32,u32
0,0


In [13]:
batter_games = batter_features.build_batter_games(raw)
batter_games.shape

# Uniqueness check on the primary key
batter_games.filter(pl.col("PA") == 0).select(["game_pk", "batter", "wOBA", "xwOBA"])

game_pk,batter,wOBA,xwOBA
i64,i64,f64,f64
745051,606466,NaN,NaN
745218,645302,NaN,NaN


In [7]:
checks = batter_games.select(
    (pl.col("PA_vL") + pl.col("PA_vR") == pl.col("PA")).all().alias("PA_split_matches_PA"),
    (pl.col("K_vL") + pl.col("K_vR") == pl.col("K")).all().alias("K_split_matches_K"),
    (pl.col("K") <= pl.col("PA")).all().alias("K_le_PA"),
    (pl.col("Chases") <= pl.col("OutZone")).all().alias("Chases_le_OutZone"),
    (pl.col("Contacts") <= pl.col("Swings")).all().alias("Contacts_le_Swings"),
    (pl.col("CSW") <= pl.col("Pitches")).all().alias("CSW_le_Pitches"),
)
checks

PA_split_matches_PA,K_split_matches_K,K_le_PA,Chases_le_OutZone,Contacts_le_Swings,CSW_le_Pitches
bool,bool,bool,bool,bool,bool
true,true,true,true,true,true


In [8]:
pa = statcast.plate_appearances(raw)
pa_check = (
    pa.group_by(["game_pk", "batter"])
    .agg(pl.len().alias("PA_check"), pl.col("is_k").sum().alias("K_check"))
)

merged = batter_games.join(pa_check, on=["game_pk", "batter"], how="left")
merged.select(
    (pl.col("PA") == pl.col("PA_check")).all().alias("PA_matches_plate_appearances"),
    (pl.col("K") == pl.col("K_check")).all().alias("K_matches_plate_appearances"),
)

PA_matches_plate_appearances,K_matches_plate_appearances
bool,bool
true,true


In [9]:
zero_pa = batter_games.filter(pl.col("PA") == 0)
zero_pa.height
zero_pa.select(["game_pk", "batter", "game_date", "Pitches", "BIP", "PA", "K"]).head(20)

game_pk,batter,game_date,Pitches,BIP,PA,K
i64,i64,date,u32,u32,u32,u32
745051,606466,2025-08-19,2,0,0,0
745218,645302,2025-08-25,2,0,0,0


In [10]:
rows_check = raw.filter(
    (pl.col("game_pk") == 745051) & (pl.col("batter") == 606466)
)
rows_check.select(["at_bat_number", "pitch_number", "description", "events", "type"])

at_bat_number,pitch_number,description,events,type
i64,i64,str,str,str
70,2,"""ball""",null,"""B"""
70,1,"""ball""",null,"""B"""


In [11]:
raw.filter((pl.col("game_pk") == 745051) & (pl.col("at_bat_number").is_between(68, 72))).select(
    ["batter", "at_bat_number", "pitch_number", "description", "events", "inning", "inning_topbot"]
).sort(["at_bat_number", "pitch_number"])

batter,at_bat_number,pitch_number,description,events,inning,inning_topbot
i64,i64,i64,str,str,i64,str
666624,68,1,"""swinging_strike""",null,8,"""Bot"""
666624,68,2,"""ball""",null,8,"""Bot"""
666624,68,3,"""ball""",null,8,"""Bot"""
666624,68,4,"""ball""",null,8,"""Bot"""
666624,68,5,"""called_strike""",null,8,"""Bot"""
…,…,…,…,…,…,…
553993,71,5,"""foul""",null,9,"""Top"""
553993,71,6,"""swinging_strike_blocked""","""strikeout""",9,"""Top"""
680728,72,1,"""ball""",null,9,"""Top"""


In [12]:
sample_batter = batter_games["batter"][0]

batter_games.filter(pl.col("batter") == sample_batter).select([
    "game_pk", "game_date", "bat_team", "opp_team", "is_home",
    "PA", "K", "BB", "HBP", "HR", "Hits",
    "PA_vL", "K_vL", "PA_vR", "K_vR",
    "Swings", "Chases", "Contacts",
    "chase_rate", "contact_rate", "swstr_rate",
    "xwOBA", "wOBA",
])

game_pk,game_date,bat_team,opp_team,is_home,PA,K,BB,HBP,HR,Hits,PA_vL,K_vL,PA_vR,K_vR,Swings,Chases,Contacts,chase_rate,contact_rate,swstr_rate,xwOBA,wOBA
i64,date,str,str,bool,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64
745039,2025-03-29,"""CHC""","""TEX""",false,3,1,0,0,0,0,0,0,3,1,6,1,4,0.25,0.666667,0.166667,0.025667,0.0
745037,2025-03-31,"""CHC""","""TEX""",false,3,2,0,0,0,0,2,2,1,0,9,2,4,0.333333,0.444444,0.357143,0.032667,0.0
746899,2025-04-02,"""CHC""","""COL""",true,3,1,0,0,0,0,0,0,3,1,5,1,4,0.5,0.8,0.125,0.223667,0.0
746897,2025-04-06,"""CHC""","""LAD""",true,4,0,0,0,0,1,0,0,4,0,12,5,11,0.454545,0.916667,0.052632,0.178,0.225
746896,2025-04-07,"""CHC""","""LAD""",true,3,0,0,0,0,1,0,0,3,0,4,1,3,0.5,0.75,0.2,0.471667,0.416667
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
746873,2025-06-06,"""CHC""","""CWS""",true,4,2,1,0,0,0,0,0,4,2,13,7,7,0.5,0.538462,0.272727,0.377033,0.175
746703,2025-06-09,"""CHC""","""CIN""",false,3,0,0,0,0,1,2,0,1,0,5,2,5,0.333333,1.0,0.0,0.679667,0.416667
745079,2025-06-13,"""CHC""","""TB""",false,1,0,0,0,0,0,1,0,0,0,1,0,1,0.0,1.0,0.0,1.478,0.0
